# Exercises: ML Fundamentals

Sharpen your scikit-learn and machine-learning workflow skills.

## Exercise 1: Manual Cross-Validation

Implement **k-fold cross-validation from scratch** (no `cross_val_score`).

**Requirements:**
- Generate a synthetic classification dataset
- Split indices manually into k=5 folds
- Train a `LogisticRegression` on each fold, record accuracy
- Compare your result with `sklearn.model_selection.cross_val_score`

In [ ]:
# YOUR CODE HERE

### Solution

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

np.random.seed(42)
X, y = make_classification(n_samples=500, n_features=10,
                            n_informative=6, random_state=42)

k = 5
indices = np.arange(len(X))
np.random.shuffle(indices)
folds = np.array_split(indices, k)

scores = []
for i in range(k):
    val_idx = folds[i]
    train_idx = np.concatenate([folds[j] for j in range(k) if j != i])
    model = LogisticRegression(max_iter=300, random_state=42)
    model.fit(X[train_idx], y[train_idx])
    acc = model.score(X[val_idx], y[val_idx])
    scores.append(acc)
    print(f"Fold {i+1}: accuracy = {acc:.4f}")

print(f"\nManual CV mean: {np.mean(scores):.4f} ± {np.std(scores):.4f}")

# Verify with sklearn
sk_scores = cross_val_score(LogisticRegression(max_iter=300, random_state=42),
                             X, y, cv=5, scoring='accuracy')
print(f"Sklearn CV mean: {sk_scores.mean():.4f} ± {sk_scores.std():.4f}")


### Explanation

We shuffle indices, split them into k equal-ish chunks, and rotate which chunk is held out. This mirrors `KFold` without stratification. Small differences from `cross_val_score` arise from different shuffle seeds.

## Exercise 2: Feature Engineering Challenge

Given raw features, **engineer new ones** that improve model performance.

**Requirements:**
- Start with a synthetic regression dataset (5 features)
- Create at least 4 new features (interactions, polynomials, binning, log)
- Show baseline vs enhanced R² scores using `Ridge`

In [ ]:
# YOUR CODE HERE

### Solution

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_regression
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import PolynomialFeatures

np.random.seed(42)
X, y = make_regression(n_samples=400, n_features=5, noise=20, random_state=42)
df = pd.DataFrame(X, columns=[f'f{i}' for i in range(5)])

# Baseline
baseline = cross_val_score(Ridge(), X, y, cv=5, scoring='r2').mean()
print(f"Baseline R²: {baseline:.4f}")

# Feature engineering
df['f0_x_f1'] = df['f0'] * df['f1']                    # interaction
df['f2_squared'] = df['f2'] ** 2                        # polynomial
df['f3_log'] = np.log1p(np.abs(df['f3']))               # log transform
df['f4_bin'] = pd.qcut(df['f4'], q=5, labels=False)     # quantile binning
df['f0_plus_f2'] = df['f0'] + df['f2']                  # additive combo

X_eng = df.values
enhanced = cross_val_score(Ridge(), X_eng, y, cv=5, scoring='r2').mean()
print(f"Enhanced R²: {enhanced:.4f}")
print(f"Improvement:  {enhanced - baseline:+.4f}")


### Explanation

Feature engineering injects domain-inspired non-linearities into a linear model. Interaction terms capture joint effects, polynomial terms capture curvature, and log transforms reduce skew. Even simple additions can boost R² when the true relationship is non-linear.

## Exercise 3: Model Comparison — Pick the Best

Train **5 different classifiers** on the same dataset and compare them.

**Requirements:**
- Models: Logistic Regression, Decision Tree, Random Forest, SVM, KNN
- Use 5-fold CV with accuracy *and* F1-score
- Print a summary table and declare a winner

In [ ]:
# YOUR CODE HERE

### Solution

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

np.random.seed(42)
X, y = make_classification(n_samples=600, n_features=15,
                            n_informative=8, n_classes=3,
                            n_clusters_per_class=1, random_state=42)

models = {
    'LogisticReg': LogisticRegression(max_iter=500, random_state=42),
    'DecisionTree': DecisionTreeClassifier(random_state=42),
    'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(random_state=42),
    'KNN': KNeighborsClassifier(),
}

results = []
for name, model in models.items():
    acc = cross_val_score(model, X, y, cv=5, scoring='accuracy')
    f1 = cross_val_score(model, X, y, cv=5, scoring='f1_macro')
    results.append({'Model': name,
                    'Accuracy': f'{acc.mean():.4f} ± {acc.std():.4f}',
                    'F1 (macro)': f'{f1.mean():.4f} ± {f1.std():.4f}',
                    '_acc_mean': acc.mean()})

results_df = pd.DataFrame(results)
winner = results_df.loc[results_df['_acc_mean'].idxmax(), 'Model']
print(results_df[['Model', 'Accuracy', 'F1 (macro)']].to_string(index=False))
print(f"\nWinner: {winner}")


### Explanation

Comparing models on the same CV folds gives a fair comparison. Accuracy works for balanced classes; F1-macro is better when class frequencies differ. Random Forest often wins on tabular data thanks to bagging and feature sub-sampling.

## Exercise 4: Pipeline Building

Build a **complete sklearn Pipeline** that handles preprocessing and modelling.

**Requirements:**
- Mix of numeric and categorical features (use `ColumnTransformer`)
- Numeric: imputation → scaling
- Categorical: imputation → one-hot encoding
- Final estimator: `RandomForestClassifier`
- Evaluate with CV

In [ ]:
# YOUR CODE HERE

### Solution

In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

np.random.seed(42)
n = 400
df = pd.DataFrame({
    'age': np.random.randint(18, 70, n).astype(float),
    'income': np.random.exponential(50000, n),
    'education': np.random.choice(['HS', 'BSc', 'MSc', 'PhD'], n),
    'region': np.random.choice(['North', 'South', 'East', 'West'], n),
})
# Inject missing values
df.loc[np.random.choice(n, 30, replace=False), 'age'] = np.nan
df.loc[np.random.choice(n, 20, replace=False), 'income'] = np.nan
df.loc[np.random.choice(n, 15, replace=False), 'education'] = np.nan

y = (df['income'].fillna(df['income'].median()) > 50000).astype(int)

num_cols = ['age', 'income']
cat_cols = ['education', 'region']

num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])
cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

preprocessor = ColumnTransformer([
    ('num', num_pipe, num_cols),
    ('cat', cat_pipe, cat_cols),
])

full_pipe = Pipeline([
    ('preprocess', preprocessor),
    ('clf', RandomForestClassifier(n_estimators=100, random_state=42)),
])

scores = cross_val_score(full_pipe, df, y, cv=5, scoring='accuracy')
print(f"Pipeline CV accuracy: {scores.mean():.4f} ± {scores.std():.4f}")


### Explanation

`ColumnTransformer` routes numeric and categorical columns through different sub-pipelines. Wrapping everything in a single `Pipeline` ensures transformations are fitted only on training data during CV, preventing data leakage.

## Exercise 5: Hyperparameter Tuning Analysis

Use `GridSearchCV` to tune a `GradientBoostingClassifier` and **analyse** how each hyperparameter affects performance.

**Requirements:**
- Tune `n_estimators`, `max_depth`, `learning_rate`
- Print best params and best score
- Show a DataFrame of all CV results sorted by mean score

In [ ]:
# YOUR CODE HERE

### Solution

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV

np.random.seed(42)
X, y = make_classification(n_samples=500, n_features=12,
                            n_informative=8, random_state=42)

param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [2, 4, 6],
    'learning_rate': [0.01, 0.1, 0.2],
}

grid = GridSearchCV(
    GradientBoostingClassifier(random_state=42),
    param_grid, cv=5, scoring='accuracy',
    return_train_score=True, n_jobs=-1,
)
grid.fit(X, y)

print(f"Best params: {grid.best_params_}")
print(f"Best CV accuracy: {grid.best_score_:.4f}")

results = (pd.DataFrame(grid.cv_results_)
           [['param_n_estimators', 'param_max_depth',
             'param_learning_rate', 'mean_test_score', 'std_test_score',
             'mean_train_score']]
           .sort_values('mean_test_score', ascending=False))
print("\nTop 10 configurations:")
print(results.head(10).to_string(index=False))


### Explanation

`GridSearchCV` exhaustively evaluates every parameter combination. Inspecting `cv_results_` reveals the train-test gap (overfitting signal) and how sensitive accuracy is to each hyperparameter.